In [1]:
import os
import sys
from pathlib import Path
ROOT = Path("../..").resolve()
os.chdir(ROOT)

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

sys.path.insert(0, str(ROOT / "experiments" / "ebm"))
os.getcwd()

'/home/user/projects/SoftStairs-QAT'

In [2]:
from softstairs_qat.utils import ReproducibilityManager, configure_logging
from loguru import logger
ReproducibilityManager().set_seed(42)
paths = configure_logging(name="ebm_qat", log_dir="experiments/ebm/logs")
logger.info("Log file: {}", paths.log_file)
BATCH_SIZE = 128
N_EPOCHS = 20
N_BITS = 8
IMG_SHAPE = (1, 28, 28)
LR = 1e-4
STRATEGIES = ["linear", "cos", "exp"]
T_START = [0.9, 0.5, 0.1]
WEIGHT_DECAYS = [0.0, 1e-4]   

TORCH_QAT_BITS = 8          # 8 или 4 (Int4WeightOnlyConfig)
TORCH_QAT_GROUP_SIZE = 32
BASELINE_RUN = f"baseline-torch-qat-int{TORCH_QAT_BITS}b-{N_EPOCHS}e"

/home/user/projects/SoftStairs-QAT/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W0826 12:53:34.563000 23394 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W0826 12:53:34.606000 23394 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
2026-08-26 12:53:34 | INFO     | __main__:<module>:5 - Log file: experiments/ebm/logs/ebm_qat_20260826_125334.log


# Check firing

In [3]:
from softstairs_qat import SoftStairsQuantizer, QuantizationConfig
def get_qconfig(strategy, t_start, total_steps, n_bits=8):
    """total_steps -> QuantizationConfig.t_step (число optimizer steps для scheduler)."""
    return QuantizationConfig(
        n_bits=n_bits,
        normalized=True,
        t_scheduler_strategy=strategy,
        t_start=t_start,
        t_end=1e-4,
        t_step=total_steps,
    )

In [4]:
import torch
from ebm_model import CNNModel, get_mnist_dataloaders

train_loader, _ = get_mnist_dataloaders(BATCH_SIZE)
total_steps = N_EPOCHS * len(train_loader)
model = CNNModel()
qconfig = get_qconfig("linear", 0.1, total_steps, N_BITS)
quantizer = SoftStairsQuantizer(
    model,
    qconfig,
    excluded_modules=set(),
    verbose=True,
)
x = torch.randn(2, 1, 28, 28)
model(x)
assert len(quantizer._hook_handles), "no handles found"
print("OK: quantizer fired on forward")

@@@ INIT cnn_layers.0.weight
@@@ INIT cnn_layers.2.weight
@@@ INIT cnn_layers.4.weight
@@@ INIT cnn_layers.6.weight
@@@ INIT cnn_layers.9.weight
@@@ INIT cnn_layers.11.weight
@@@ FIRED cnn_layers.0.weight
@@@ FIRED cnn_layers.2.weight
@@@ FIRED cnn_layers.4.weight
@@@ FIRED cnn_layers.6.weight
@@@ FIRED cnn_layers.9.weight
@@@ FIRED cnn_layers.11.weight
OK: quantizer fired on forward


# baseline no-qat

In [ ]:
import torch
from torchao.quantization import quantize_, Int8WeightOnlyConfig, Int4WeightOnlyConfig
from torchao.quantization.qat import QATConfig
from torchao.quantization.granularity import PerGroup, PerRow
from ebm_model import DeepEnergyModel, fit_ebm_model, get_mnist_dataloaders, compute_snapshot_steps, save_gradient_heatmaps

class TorchQATDeepEnergyModel(DeepEnergyModel):
    """Baseline: нативная QAT через torchao (prepare на CNN)."""
    def __init__(self, torch_qconfig, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self._torch_qconfig = torch_qconfig
        self._torch_qat_prepared = False
    def setup(self, stage=None):
        super().setup(stage)
        if not self._torch_qat_prepared:
            quantize_(self.cnn, QATConfig(self._torch_qconfig, step="prepare"))
            self._torch_qat_prepared = True
            logger.info("Torch QAT prepared on CNN")

    def optimizer_step(self, epoch, batch_idx, optimizer, optimizer_closure):
        optimizer_closure()
        torch.nn.utils.clip_grad_norm_(self.cnn.parameters(), max_norm=0.1)
        optimizer.step()
        optimizer.zero_grad(set_to_none=True)
        if self.global_step in self.snapshot_steps:
            grads = {
                n: p.grad.detach().clone()
                for n, p in self.cnn.named_parameters()
                if p.grad is not None
            }
            save_gradient_heatmaps(grads, self.global_step, self.grad_dir)
            
    def on_train_epoch_end(self):
        super().on_train_epoch_end()

def get_torch_qat_config(n_bits=8, group_size=32):
    if n_bits == 8:
        if group_size is None:
            return Int8WeightOnlyConfig(granularity=PerRow())  # per-channel
        return Int8WeightOnlyConfig(granularity=PerGroup(group_size))
    if n_bits == 4:
        if group_size is None:
            raise ValueError("Int4 QAT usually needs group_size, e.g. 32")
        return Int4WeightOnlyConfig(granularity=PerGroup(group_size))
    raise ValueError(f"Unsupported n_bits={n_bits}")


In [6]:
train_loader, val_loader = get_mnist_dataloaders(BATCH_SIZE)
steps_per_epoch = len(train_loader)
torch_qconfig = get_torch_qat_config(TORCH_QAT_BITS, TORCH_QAT_GROUP_SIZE)

model = TorchQATDeepEnergyModel(
    torch_qconfig,
    img_shape=IMG_SHAPE,
    batch_size=BATCH_SIZE,
    lr=LR,
    weight_decay=0.0,
    run_name=BASELINE_RUN,
    snapshot_steps=compute_snapshot_steps(N_EPOCHS, steps_per_epoch),
    qconfig=None,          # SoftStairs не используем
    quantizer=None,
)

fit_ebm_model(model, train_loader, val_loader, max_epochs=N_EPOCHS)

AssertionError: Only support version 2 with group_size=None, got 32. Use granularity=PerGroup(32) instead.

# SoftStairs QAT

In [10]:
import csv
from pathlib import Path
from loguru import logger
from softstairs_qat import SoftStairsQuantizer
from ebm_model import EBMTrainer, get_mnist_dataloaders, compute_snapshot_steps


def run_experiment_nb(n_epochs, strategy, t_start, n_bits=8, weight_decay=0.0, log_every_n_steps=50):
    train_loader, val_loader = get_mnist_dataloaders(BATCH_SIZE)
    steps_per_epoch = len(train_loader)
    total_steps = n_epochs * steps_per_epoch

    qconfig = get_qconfig(strategy, t_start, total_steps, n_bits)
    run_name = f"{strategy}-{t_start}-{n_bits}b-{n_epochs}e-wd{weight_decay}"

    logger.info(
        "Starting run={} strategy={} t_start={} bits={} wd={} total_steps={}",
        run_name, strategy, t_start, n_bits, weight_decay, total_steps,
    )

    class QATTrainer(EBMTrainer):
        def __init__(self, qconfig, log_every_n_steps=50, *args, **kwargs):
            super().__init__(*args, **kwargs)
            self._qconfig = qconfig
            self._log_every = log_every_n_steps
            self._step_log_path = self.out_dir / "t_schedule.csv"
            self._step_log_initialized = False

        def build_model(self):
            cnn = super().build_model()
            self.quantizer = SoftStairsQuantizer(
                cnn,
                self._qconfig,
                excluded_modules=set(),
            )
            logger.info("Quantized layers: {}", sorted(self.quantizer._scales.keys()))
            return cnn

        def _log_step_t(self):
            if self.quantizer is None:
                return
            write_header = not self._step_log_initialized
            with self._step_log_path.open("a", newline="") as f:
                writer = csv.DictWriter(f, fieldnames=["step", "t", "epoch"])
                if write_header:
                    writer.writeheader()
                    self._step_log_initialized = True
                writer.writerow({
                    "step": self.global_step,
                    "t": self.quantizer.get_current_t(),
                    "epoch": getattr(self, "_current_epoch", -1),
                })

        def optimizer_step(self):
            super().optimizer_step()
            if self.quantizer is not None and self.global_step % self._log_every == 0:
                logger.debug("step={} t={:.6f}", self.global_step, self.quantizer.get_current_t())
                self._log_step_t()

        def train_one_epoch(self, train_loader, epoch):
            self._current_epoch = epoch
            return super().train_one_epoch(train_loader, epoch)

    trainer = QATTrainer(
        qconfig,
        log_every_n_steps=log_every_n_steps,
        img_shape=IMG_SHAPE,
        batch_size=BATCH_SIZE,
        lr=LR,
        weight_decay=weight_decay,
        run_name=run_name,
        snapshot_steps=compute_snapshot_steps(n_epochs, steps_per_epoch),
    )

    trainer.train(train_loader, val_loader, n_epochs=n_epochs)
    logger.info("Finished run={}", run_name)
    return run_name

# Running

In [14]:
run_experiment_nb(
    n_epochs=20,
    strategy="step",
    t_start=0.9,
    n_bits=4,
    weight_decay=0.0,
)

2026-08-25 17:18:07 | INFO     | __main__:run_experiment_nb:16 - Starting run=step-0.9-4b-20e-wd0.0 strategy=step t_start=0.9 bits=4 wd=0.0 total_steps=9360
2026-08-25 17:18:07 | INFO     | __main__:build_model:36 - Quantized layers: ['cnn_layers.0.weight', 'cnn_layers.11.weight', 'cnn_layers.2.weight', 'cnn_layers.4.weight', 'cnn_layers.6.weight', 'cnn_layers.9.weight']


[step-0.9-4b-20e-wd0.0] epoch 1/20 loss=0.0028 val_cdiv=-0.0003 P=0.000 R=0.000, t=0.0001
[step-0.9-4b-20e-wd0.0] epoch 2/20 loss=0.0032 val_cdiv=-0.0004 P=0.000 R=0.000, t=0.0001
[step-0.9-4b-20e-wd0.0] epoch 3/20 loss=0.0027 val_cdiv=-0.0005 P=0.000 R=0.000, t=0.0001
[step-0.9-4b-20e-wd0.0] epoch 4/20 loss=0.0021 val_cdiv=-0.0006 P=0.000 R=0.000, t=0.0001
[step-0.9-4b-20e-wd0.0] epoch 5/20 loss=0.0016 val_cdiv=-0.0008 P=0.000 R=0.000, t=0.0001
[step-0.9-4b-20e-wd0.0] epoch 6/20 loss=0.0012 val_cdiv=-0.0009 P=1.000 R=0.001, t=0.0001
[step-0.9-4b-20e-wd0.0] epoch 7/20 loss=0.0009 val_cdiv=-0.0011 P=1.000 R=0.020, t=0.0001
[step-0.9-4b-20e-wd0.0] epoch 8/20 loss=0.0006 val_cdiv=-0.0013 P=1.000 R=0.192, t=0.0001
[step-0.9-4b-20e-wd0.0] epoch 9/20 loss=0.0003 val_cdiv=-0.0015 P=1.000 R=0.435, t=0.0001
[step-0.9-4b-20e-wd0.0] epoch 10/20 loss=0.0000 val_cdiv=-0.0018 P=1.000 R=0.647, t=0.0001
[step-0.9-4b-20e-wd0.0] epoch 11/20 loss=-0.0002 val_cdiv=-0.0022 P=1.000 R=0.786, t=0.0001
[step-0

2026-08-25 17:35:10 | INFO     | __main__:run_experiment_nb:76 - Finished run=step-0.9-4b-20e-wd0.0


[step-0.9-4b-20e-wd0.0] epoch 20/20 loss=-0.0017 val_cdiv=-0.0065 P=1.000 R=0.874, t=0.0001


'step-0.9-4b-20e-wd0.0'

In [ ]:
from itertools import product

for strat, t, wd in product(STRATEGIES, T_START, WEIGHT_DECAYS):
    print("Running:", strat, t, "wd=", wd)
    try:
        run_experiment_nb(
            n_epochs=N_EPOCHS,
            strategy=strat,
            t_start=t,
            n_bits=N_BITS,
            weight_decay=wd,
        )
    except KeyboardInterrupt:
        raise
    except Exception as e:
        failed_log = ROOT / "experiments" / "ebm" / "failed_runs.txt"
        with failed_log.open("a") as f:
            f.write(f"FAILED {strat} {t} wd={wd}: {e}\n")
        print("FAILED:", e)